# Answer Equivalence
Test whether two answers (e.g., the LLM's and Eedi's) are semantically equivalent under LLM-as-a-judge.
Also evaluate the performance of LLM-as-a-judge.

Supports Appendix A.2 ("Equivalence checks") and the main-text judge numbers in Section 4 ("Performance metrics"): gpt-4-1-mini precision/recall for the distractor-equivalence judge, validated manually over 500 Eedi + 500 SciQ comparisons.

In [ ]:
import pandas as pd
import os
import glob
import json
import random
from openai import OpenAI
from tqdm import tqdm

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

from src.datasets import get_or_create_dataset
from src.equality import MathSemanticEqualityChecker, ScienceSemanticEqualityChecker
from src.model_configurations import gpt_4_1_mini_det_config

In [ ]:
equality_model_config = gpt_4_1_mini_det_config
equality_client = OpenAI(base_url=equality_model_config["base_url"], api_key=os.environ.get(equality_model_config["api_key_var"], None))
math_semantic_equality_checker = MathSemanticEqualityChecker(equality_client, equality_model_config)

## Eedi

### Agreement with Human Annotations

For Eedi, we do this on the student simulation traces, so you need to run the error-simulation notebook first.

In [ ]:
df = pd.read_csv("eedi_data/sim_results/annotated/naive-deepseek-reasoner.csv")
eedi_dataset = get_or_create_dataset("eedi_data")

In [ ]:
gpt_4_1_labels = []
for i,row in tqdm(list(df.iterrows())):
    lbl = math_semantic_equality_checker.is_equal(row["problem"], row["groundtruth"], row["llm"])
    gpt_4_1_labels.append(lbl)

In [ ]:
y_true = df["match_annot"].astype(bool).to_numpy()
y_pred = [bool(x) for x in gpt_4_1_labels]

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
# -> Appendix A.2: "98% precision and 92% recall ... over 500 Eedi comparisons"

## SciQ

In [ ]:
sciq_dataset = get_or_create_dataset("sciq_data", n_limit=500)

science_equivalence_check_path = "cache/science_semantic_equivalence_checker.pkl"
if os.path.exists(science_equivalence_check_path):
    print("Loading existing science semantic equality checker")
    science_semantic_equality_checker = ScienceSemanticEqualityChecker.load(equality_client, science_equivalence_check_path)
else:
    science_semantic_equality_checker = ScienceSemanticEqualityChecker(equality_client, equality_model_config)
    os.makedirs("cache", exist_ok=True)
    science_semantic_equality_checker.save(science_equivalence_check_path)

In [ ]:
# Label all (correct_answer, model_distractor) pairs from deepseek + glm-4.7 joint_results.
# We keep the checker's memoization on disk so later performance runs can reuse these labels.
# Two CSVs are written (source of the manually-annotated 500 SciQ comparisons in Appendix A.2):
#   - predicted_positives.csv: ALL rows flagged as equivalent by the LLM judge (for precision annotation)
#   - predicted_negatives_sample.csv: random subset of predicted negatives (for recall annotation)
random.seed(42)

N_NEGATIVES_FOR_RECALL = 500  # bump this if the annotated subset yields too few false negatives

joint_dir = "sciq_data/joint_results"
family_globs = {
    "deepseek": os.path.join(joint_dir, "*deepseek*_responses_by_datapointid.json"),
    "glm-4.7": os.path.join(joint_dir, "*glm-4.7*_responses_by_datapointid.json"),
}

def collect_pairs(family: str, file_glob: str) -> list[dict]:
    pairs = []
    for path in sorted(glob.glob(file_glob)):
        setting = os.path.basename(path).replace("_responses_by_datapointid.json", "")
        with open(path, "r") as f:
            responses = json.load(f)
        for dpid, resp in responses.items():
            problem = sciq_dataset[int(dpid)]["Problem"]
            for i in range(1, problem["NumDistractors"] + 1):
                key = f"distractor{i}_answer"
                if key not in resp:
                    continue
                pairs.append({
                    "family": family,
                    "setting": setting,
                    "datapoint_id": dpid,
                    "distractor_idx": i,
                    "question": problem["Question"],
                    "correct_answer": problem["Answer"],
                    "model_answer": resp[key],
                })
    return pairs

all_pairs = []
for family, g in family_globs.items():
    pairs = collect_pairs(family, g)
    all_pairs.extend(pairs)
    print(f"{family}: {len(pairs)} pairs")

eval_df = pd.DataFrame(all_pairs)
print(f"Total pairs to label: {len(eval_df)}")

predictions = []
save_every = 200
for idx, row in enumerate(tqdm(list(eval_df.itertuples(index=False)))):
    predictions.append(
        science_semantic_equality_checker.is_equal(row.question, row.correct_answer, row.model_answer)
    )
    if (idx + 1) % save_every == 0:
        science_semantic_equality_checker.save(science_equivalence_check_path)

science_semantic_equality_checker.save(science_equivalence_check_path)

eval_df["match_llm"] = predictions
eval_df["match_annot"] = ""  # to be filled in manually

os.makedirs("sciq_data/equivalence_eval", exist_ok=True)

positives_df = eval_df[eval_df["match_llm"]].reset_index(drop=True)
negatives_df = eval_df[~eval_df["match_llm"]].reset_index(drop=True)
negatives_sample_df = negatives_df.sample(n=min(N_NEGATIVES_FOR_RECALL, len(negatives_df)), random_state=42).reset_index(drop=True)

all_csv_path = "sciq_data/equivalence_eval/science_equivalence_eval_all.csv"
positives_csv_path = "sciq_data/equivalence_eval/predicted_positives.csv"
negatives_csv_path = "sciq_data/equivalence_eval/predicted_negatives_sample.csv"
positives_df.to_csv(positives_csv_path, index=False)
negatives_sample_df.to_csv(negatives_csv_path, index=False)
eval_df.to_csv(all_csv_path, index=False)

print(f"Predicted positives: {len(positives_df)} → {positives_csv_path}")
print(f"Predicted negatives (sampled {len(negatives_sample_df)} of {len(negatives_df)}): {negatives_csv_path}")

In [ ]:
# compute stats (we subsampled the negatives => need to rescale)
annot_pos_df = pd.read_csv("sciq_data/equivalence_eval/predicted_positives.csv")
annot_neg_df = pd.read_csv("sciq_data/equivalence_eval/predicted_negatives_sample.csv")
all_df = pd.read_csv("sciq_data/equivalence_eval/science_equivalence_eval_all.csv")

def parse_bool(x):
    if isinstance(x, bool): return x
    if pd.isna(x): return None
    s = str(x).strip().lower()
    if s in {"true", "t", "1", "yes", "y"}: return True
    if s in {"false", "f", "0", "no", "n"}: return False
    return None

annot_pos_df["match_annot"] = annot_pos_df["match_annot"].map(parse_bool)
annot_neg_df["match_annot"] = annot_neg_df["match_annot"].map(parse_bool)

missing_pos = annot_pos_df["match_annot"].isna().sum()
missing_neg = annot_neg_df["match_annot"].isna().sum()
if missing_pos or missing_neg:
    print(f"Warning: missing annotations — {missing_pos} positives, {missing_neg} negatives. Dropping.")
annot_pos_df = annot_pos_df.dropna(subset=["match_annot"])
annot_neg_df = annot_neg_df.dropna(subset=["match_annot"])

tp = int(annot_pos_df["match_annot"].sum())
fp = int((~annot_pos_df["match_annot"].astype(bool)).sum())

n_neg_total = int((~all_df["match_llm"].astype(bool)).sum())
n_neg_sampled = len(annot_neg_df)
fn_in_sample = int(annot_neg_df["match_annot"].sum())
tn_in_sample = n_neg_sampled - fn_in_sample

scale = n_neg_total / n_neg_sampled
fn_est = fn_in_sample * scale

precision = tp / (tp + fp)
recall = tp / (tp + fn_est)


print(f"Annotated predicted positives: {len(annot_pos_df)}  (TP={tp}, FP={fp})")
print(f"Annotated predicted negatives: {n_neg_sampled} / {n_neg_total}  (FN_in_sample={fn_in_sample}, TN_in_sample={tn_in_sample})")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
# -> Appendix A.2: "97% precision and 100% recall ... over 500 SciQ comparisons"